In [ ]:
%pip install xgboost==1.7.3
%pip install optuna

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import random
import time
import itertools
import optuna
import gc
import pickle
import tracemalloc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Sklearn Imports
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Tắt log & Cấu hình hiển thị
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# ===================================================================
# 0. SETUP & LOAD DATA
# ===================================================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
SEED = 42
seed_everything(SEED)

BASE = '/content/drive/MyDrive/Colab Notebooks/data/raw_images'

DATA_SOURCES = {
    'haze': os.path.join(BASE, 'Haze'),
    'rain': os.path.join(BASE, 'Rain'),
    'shine': os.path.join(BASE, 'Shine')
}
IMG_SIZE = 256
HOG_PPC = 16 

img_paths, labels_raw = [], []
for label, path in DATA_SOURCES.items():
    if os.path.exists(path):
        files = sorted([os.path.join(path, f) for f in os.listdir(path) if f.lower().endswith(('.jpg','.png','.jpeg'))])
        img_paths.extend(files)
        labels_raw.extend([label] * len(files))

le = LabelEncoder()
y_encoded = le.fit_transform(labels_raw)
class_names = le.classes_

# ===================================================================
# 1. FEATURE EXTRACTION
# ===================================================================
def get_color_stats(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    return [np.mean(hsv[:,:,i]) for i in range(3)] + [np.std(hsv[:,:,i]) for i in range(3)]
def get_sobel_stats(gray):
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sx**2 + sy**2)
    return [np.mean(mag), np.var(mag)]
def get_lbp_hist(gray):
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10))
    hist = hist.astype("float"); return hist / (hist.sum() + 1e-7)
def get_hog_stats(gray):
    hog_v = hog(gray, orientations=12, pixels_per_cell=(HOG_PPC, HOG_PPC), cells_per_block=(2,2), visualize=False, feature_vector=True)
    return [np.mean(hog_v), np.std(hog_v), np.max(hog_v)]
def get_glcm_stats(gray):
    glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/2], levels=256, symmetric=True, normed=True)
    contrast = graycoprops(glcm, 'contrast').flatten()
    correlation = graycoprops(glcm, 'correlation').flatten()
    energy = graycoprops(glcm, 'energy').flatten()
    homogeneity = graycoprops(glcm, 'homogeneity').flatten()
    return np.concatenate([contrast, correlation, energy, homogeneity])

def extract_all(paths):
    l_c, l_s, l_l, l_h, l_glcm = [], [], [], [], []
    print("[1] Extracting features...")
    for path in tqdm(paths):
        img = cv2.imread(path); img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        l_c.append(get_color_stats(img))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        l_s.append(get_sobel_stats(gray)); l_l.append(get_lbp_hist(gray))
        l_h.append(get_hog_stats(gray)); l_glcm.append(get_glcm_stats(gray))
    return {'Color': np.array(l_c), 'Sobel': np.array(l_s), 'LBP': np.array(l_l), 'HOG': np.array(l_h), 'GLCM': np.array(l_glcm)}

all_features = extract_all(img_paths)

indices = np.arange(len(img_paths))
X_train_idx, X_test_idx, y_train, y_test = train_test_split(indices, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded)
# CV Strategy
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

base_features = ['Color', 'Sobel', 'LBP', 'HOG', 'GLCM']
all_combinations = []
for r in range(1, len(base_features) + 1):
    for combo in itertools.combinations(base_features, r):
        all_combinations.append({'name': " + ".join(combo), 'components': list(combo)})

def plot_cm(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
    plt.title(title, fontsize=10); plt.ylabel('True Label'); plt.xlabel('Predicted Label'); plt.tight_layout(); plt.show()

# ===================================================================
# 2. MAIN LOOP: XGBOOST (DEFAULT vs TUNED + WEIGHT DECISION)
# ===================================================================
results = []
checkpoint_file = 'xgb_smart_tuning_results.csv'

print("\n" + "="*80)
print(f" STARTING EXPERIMENT")
print(f" 1. Calc Default CV | 2. Tune Params & Weight | 3. Final Test & Latency")
print("="*80)

for idx, combo in enumerate(all_combinations):
    combo_name = combo['name']
    comps = combo['components']
    
    X_train_curr = np.hstack([all_features[c][X_train_idx] for c in comps])
    X_test_curr  = np.hstack([all_features[c][X_test_idx] for c in comps])
    
    print(f"\n{'#'*60}")
    print(f" [{idx+1}/31] COMBO: {combo_name}")
    print(f"{'#'*60}")
    
    # --- PHASE A: CALCULATE DEFAULT CV SCORE (BASELINE) ---
    # XGBoost mặc định, không chỉnh tham số, không weight
    xgb_default = XGBClassifier(eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
    pipe_default = Pipeline([('scaler', StandardScaler()), ('clf', xgb_default)])
    
    # Đo CV trên tập Train
    scores_default = cross_val_score(pipe_default, X_train_curr, y_train, cv=cv_strategy, scoring='f1_macro', n_jobs=-1)
    default_cv_f1 = scores_default.mean()
    
    print(f"   [Baseline] Default CV F1: {default_cv_f1:.4f}")

    # --- PHASE B: OPTUNA TUNING (PARAMS + WEIGHT) ---
    def objective(trial):
        # 1. Tune XGBoost Params
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 5),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'eval_metric': 'mlogloss', 'random_state': SEED, 'n_jobs': -1
        }
        
        # 2. Tune Balancing Strategy (Có dùng Weight hay không?)
        use_balancing = trial.suggest_categorical('use_balancing', [True, False])
        
        # 3. Manual CV Loop
        cv_scores = []
        for train_idx_cv, val_idx_cv in cv_strategy.split(X_train_curr, y_train):
            X_tr_fold, X_val_fold = X_train_curr[train_idx_cv], X_train_curr[val_idx_cv]
            y_tr_fold, y_val_fold = y_train[train_idx_cv], y_train[val_idx_cv]
            
            sample_w = None
            if use_balancing:
                sample_w = compute_sample_weight('balanced', y_tr_fold)
            
            scaler = StandardScaler()
            X_tr_fold_sc = scaler.fit_transform(X_tr_fold)
            X_val_fold_sc = scaler.transform(X_val_fold)
            
            clf = XGBClassifier(**params)
            clf.fit(X_tr_fold_sc, y_tr_fold, sample_weight=sample_w)
            
            preds = clf.predict(X_val_fold_sc)
            cv_scores.append(f1_score(y_val_fold, preds, average='macro'))
            
        return np.mean(cv_scores)

    study = optuna.create_study(direction='maximize')
    
    # Enqueue Default Params (Để đảm bảo Tuned >= Default)
    study.enqueue_trial({
        'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3, 'subsample': 1.0, 
        'colsample_bytree': 1.0, 'min_child_weight': 1, 'gamma': 0, 'use_balancing': False
    })
    
    study.optimize(objective, n_trials=50) 
    
    best_params = study.best_params
    tuned_cv_f1 = study.best_value
    
    # --- PHASE C: FINAL TRAIN & TEST PREDICT (MEASURE LATENCY) ---
    use_balancing_final = best_params.pop('use_balancing')
    final_xgb = XGBClassifier(**best_params, eval_metric='mlogloss', random_state=SEED, n_jobs=-1)
    
    final_weights = None
    if use_balancing_final:
        final_weights = compute_sample_weight('balanced', y_train)

    scaler_final = StandardScaler()
    X_train_sc = scaler_final.fit_transform(X_train_curr)
    X_test_sc = scaler_final.transform(X_test_curr)
    
    # Train
    final_xgb.fit(X_train_sc, y_train, sample_weight=final_weights)
    
    # Measure Latency
    t0 = time.time()
    y_test_pred = final_xgb.predict(X_test_sc)
    t1 = time.time()
    
    # Calculate Latency (ms per image)
    latency_ms = ((t1 - t0) * 1000) / len(X_test_curr)
    
    test_f1 = f1_score(y_test, y_test_pred, average='macro')
    
    # --- REPORT ---
    print(f"   >>> Result: Def_CV={default_cv_f1:.4f} | Tuned_CV={tuned_cv_f1:.4f} | Test_F1={test_f1:.4f}")
    print(f"   >>> Latency: {latency_ms:.2f} ms/img | Mode: {'Weighted' if use_balancing_final else 'Unweighted'}")
    print("-" * 50)
    print(classification_report(y_test, y_test_pred, target_names=class_names, digits=4))
    
    plot_title = f"XGB | {combo_name}\nMode: {'Weighted' if use_balancing_final else 'Unweighted'} | F1: {test_f1:.4f}"
    plot_cm(y_test, y_test_pred, plot_title)
    print("-" * 50)

    results.append({
        'Combination': combo_name,
        'Default CV F1': round(default_cv_f1, 4), # <--- Đã thêm cột này
        'Tuned CV F1': round(tuned_cv_f1, 4),
        'Test F1 (Final)': round(test_f1, 4),
        'Improvement': round(tuned_cv_f1 - default_cv_f1, 4),
        'Latency (ms/img)': round(latency_ms, 4), # <--- Đã thêm cột này
        'Balancing Mode': 'Weighted' if use_balancing_final else 'Unweighted',
        'Best Params': str(best_params)
    })
    
    pd.DataFrame(results).to_csv(checkpoint_file, index=False)
    gc.collect()

df_final = pd.DataFrame(results)
print("\n>>> RESULTS SUMMARY:")
cols = ['Combination', 'Default CV F1', 'Tuned CV F1', 'Test F1 (Final)', 'Latency (ms/img)']
print(df_final.sort_values(by='Test F1 (Final)', ascending=False)[cols].head(10).to_string(index=False))